# tk-assistant · QLoRA-обучение на T4 (Этап 3)

**Runtime → Change runtime type → T4 GPU**, затем ячейки сверху вниз.

Эксперимент №1: epochs=1, lr=2e-4. Для №2/№3 — скопируй конфиг, поменяй `epochs` и `output.adapter_dir`.
Трекинг — TensorBoard (локально, без ключей). HF_TOKEN нужен только для пуша на HF Hub.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install unsloth tensorboard

## 1. Свежий клон репозитория + чистое окружение

В выводе проверь: последняя версия коммита и `report_to: tensorboard`.

In [ ]:
%cd /content
!rm -rf tk-assistant
!git clone https://github.com/markkentos/tk-assistant.git
%cd /content/tk-assistant
!git log --oneline -1
!grep report_to configs/train_qlora.yaml
import os
os.environ.pop("WANDB_API_KEY", None)
print("окружение почищено")

## 2. Обучение (~15–25 минут, 158 шагов)

В конце — 10 пробных ответов tuned-модели и строка «Адаптер сохранён».

In [ ]:
!python src/train_qlora.py --config configs/train_qlora.yaml

## 2.1 Графики обучения (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/adapter_e1_lr2e-4/checkpoints

## 3. Скачать адаптер (понадобится для HF/GGUF на этапе 5)

In [ ]:
!zip -r adapter_e1_lr2e-4.zip outputs/adapter_e1_lr2e-4 -x "*/checkpoints/*"
from google.colab import files
files.download("adapter_e1_lr2e-4.zip")

## 4. Ответы tuned-модели на бенчмарке (той же сессией, ~15 минут)

В конце скачается `answers_tuned.jsonl` — положи его в `eval/runs/` проекта.

In [ ]:
!python src/generate_answers.py --mode tuned --adapter outputs/adapter_e1_lr2e-4 --input eval/benchmark.jsonl --output eval/runs/answers_tuned.jsonl
files.download("eval/runs/answers_tuned.jsonl")